# Physics-Guided Conditional Diffusion for Accelerated MRI Reconstruction

Colab notebook. Run the cells in order.

**Before you start:** set the runtime to a GPU. *Runtime -> Change runtime type -> T4 GPU*.

Sections 1-4 need no dataset. Section 5 onward needs fastMRI.

## 1. Setup

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# Upload the project zip, or clone your own repo instead.
from google.colab import files
uploaded = files.upload()   # choose mri_diffusion_project.zip

In [ ]:
!unzip -q -o mri_diffusion_project.zip
%cd mri_diffusion_project
!pip install -q -r requirements.txt
!ls

## 2. Verify the pipeline

No dataset required. If anything here fails, stop and fix it before training.

In [ ]:
!python scripts/sanity_check.py

In [ ]:
!python -m pytest tests/ -q

## 3. Prove it learns, on synthetic phantoms

Takes a few minutes on a GPU. These are ellipse phantoms, not anatomy: this checks
the machinery end to end, it is not a result you can report.

In [ ]:
!python scripts/make_synthetic_data.py --out data/synthetic --train 8 --val 3 --slices 16 --size 64
!python scripts/train.py --config configs/synthetic_cpu.yaml \
    --train_dir data/synthetic/train --val_dir data/synthetic/val --device cuda

## 4. Look at the training curve

In [ ]:
import json
import matplotlib.pyplot as plt

with open('results/train_history.json') as f:
    h = json.load(f)

epochs = [r['epoch'] for r in h['history']]
psnr = [r['psnr'] for r in h['history']]

plt.figure(figsize=(7, 4))
plt.plot(epochs, psnr, marker='o', label='diffusion + DC')
plt.axhline(h['baseline_psnr'], ls='--', c='gray', label='zero-filled baseline')
plt.xlabel('epoch'); plt.ylabel('validation PSNR (dB)')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 5. fastMRI data

There is no public direct download link. Register at
[fastmri.med.nyu.edu](https://fastmri.med.nyu.edu/), accept the data-use agreement,
and NYU emails you personal, time-limited URLs.

You want the **single-coil knee** files. The full training set is ~90 GB, which does
not fit a free Colab session: download a subset, or mount Drive and keep a subset there.

Expected layout:
```
data/fastmri/train/*.h5
data/fastmri/val/*.h5
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point these at wherever your .h5 files live.
TRAIN_DIR = '/content/drive/MyDrive/fastmri/singlecoil_train'
VAL_DIR   = '/content/drive/MyDrive/fastmri/singlecoil_val'

!ls "$TRAIN_DIR" | head
!ls "$TRAIN_DIR" | wc -l

## 6. Cache the slices

Strongly recommended. The raw path runs a 640x372 inverse FFT for every slice, every
epoch; caching turns that into a memmap read. Run once, reuse every session.

Roughly 0.4 MB per slice, so 3000 slices is about 1.2 GB.

In [ ]:
!python scripts/prepare_data.py --src "$TRAIN_DIR" --out data/cache/train.npy --max_files 60
!python scripts/prepare_data.py --src "$VAL_DIR"   --out data/cache/val.npy   --max_files 12

In [ ]:
# Tell the config to use the cache.
import yaml

cfg = yaml.safe_load(open('configs/colab_fast.yaml'))
cfg['train_cache'] = 'data/cache/train.npy'
cfg['val_cache']   = 'data/cache/val.npy'
yaml.safe_dump(cfg, open('configs/colab_run.yaml', 'w'), sort_keys=False)
print(open('configs/colab_run.yaml').read())

## 6b. Diagnose the data BEFORE training (do not skip)

A healthy zero-filled image scores far above an all-zero image. If it does not, no
amount of training will help and you will waste the session. This prints the file
layout, the zero-filled quality at 4x and 8x, and a plain verdict.

**If it prints [BAD], stop and fix the data first.** Training will also refuse to start
if the zero-filled baseline is below 15 dB.

In [ ]:
!python scripts/diagnose_data.py --dir "$VAL_DIR" --config configs/colab_run.yaml --samples 40

from IPython.display import Image, display
display(Image('results/diagnose/look_at_this.png'))   # LOOK at it: target, zero-filled, error, mask

## 7. Train at 4x

Watch the `vs ZF` column. That is the only number that matters early on: it is the
gain over the zero-filled baseline, and it should turn positive and keep rising.

Checkpoints go to `checkpoints/best.pt` and `checkpoints/latest.pt`. Copy them to
Drive before the session times out.

In [ ]:
!python scripts/train.py --config configs/colab_run.yaml \
    --train_dir "$TRAIN_DIR" --val_dir "$VAL_DIR" --device cuda

In [ ]:
!mkdir -p /content/drive/MyDrive/mri_checkpoints
!cp checkpoints/best.pt /content/drive/MyDrive/mri_checkpoints/

# To resume in a later session:
# !python scripts/train.py --config configs/colab_run.yaml \
#     --train_dir "$TRAIN_DIR" --val_dir "$VAL_DIR" --device cuda \
#     --resume /content/drive/MyDrive/mri_checkpoints/best.pt

## 8. Evaluate

Reports the model and the zero-filled baseline on identical masks, plus the spread
across slices, and writes side-by-side figures.

In [ ]:
!python scripts/evaluate.py --config configs/colab_run.yaml \
    --checkpoint checkpoints/best.pt --val_dir "$VAL_DIR" --samples 100 --device cuda

In [ ]:
from IPython.display import Image, display
import glob

for path in sorted(glob.glob('results/figures/*.png'))[:4]:
    display(Image(path))

## 9. Ablation

This is what makes the project defensible: it shows which component actually helps
rather than asserting it.

In [ ]:
!python scripts/ablation.py --config configs/colab_run.yaml \
    --checkpoint checkpoints/best.pt --val_dir "$VAL_DIR" --samples 100 --device cuda

## 10. 8x, mixed, and refinement

Only move on once 4x beats the baseline. Adding complexity on top of a stage that is
not yet working makes the ablation uninterpretable: you can no longer tell which part
helped.

In [ ]:
# 8x acceleration
!python scripts/train.py --config configs/accel8.yaml \
    --train_dir "$TRAIN_DIR" --val_dir "$VAL_DIR" --device cuda \
    --output_dir checkpoints/accel8

In [ ]:
# Mixed 4x + 8x
!python scripts/train.py --config configs/mixed.yaml \
    --train_dir "$TRAIN_DIR" --val_dir "$VAL_DIR" --device cuda \
    --output_dir checkpoints/mixed

In [ ]:
# Stage 2: refinement U-Net, diffusion model frozen
!python scripts/train_refinement.py --config configs/colab_run.yaml \
    --train_dir "$TRAIN_DIR" --val_dir "$VAL_DIR" \
    --diffusion_checkpoint checkpoints/best.pt --device cuda

In [ ]:
# Final table, with the refinement row included
!python scripts/ablation.py --config configs/colab_run.yaml \
    --checkpoint checkpoints/best.pt \
    --refinement checkpoints/refinement_best.pt \
    --val_dir "$VAL_DIR" --samples 100 --device cuda

## Troubleshooting

**CUDA out of memory.** Change one thing at a time in the config: `batch_size: 1`
with `gradient_accumulation: 8`, then `base_channels: 32`, then
`attention_levels: [3]`.

**An epoch takes forever.** Validation runs the full sampler, so it dominates. Lower
`val_max_batches` and `inference_steps`. Also make sure you are using the cache.

**PSNR is below the baseline for the first few epochs.** Expected. The sampler starts
from noise, and an untrained network leaves that noise in the unmeasured lines. What
matters is that the `vs ZF` gap closes steadily.

**PSNR is flat and the diffusion loss is tiny from the start.** The classic symptom of
a mis-scaled residual. Check the `residual_scale` printed at startup: it should be
roughly 0.05-0.3. If you pinned `residual_scale` in the config, set it back to `0` to
re-estimate it.

**DC error is around 1e-2 instead of 1e-7.** Something is modifying the image after the
projection. Check for a stray `.clamp()` after `data_consistency()`.